<a href="https://colab.research.google.com/github/VickyW2366/Shors-optimisations/blob/main/Optimization_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#!pip install qiskit
#!pip install qiskit-aer
import qiskit
from qiskit import QuantumCircuit, transpile
from qiskit_aer import Aer
from qiskit.circuit.library import QFT
from qiskit.visualization import plot_histogram
from math import gcd
from fractions import Fraction
from sympy.ntheory import factorint
import random
from scipy.stats import norm
import numpy as np
import time

#Adaptive Shots Based on Measurement Statistics
#Dynamically adjust number of shots based on observed entropy.

# Measures how long the program takes to run
start_time = time.monotonic()

def run_shor_circuit(N, a, shots):
    """
    Constructs and runs a simplified Shor's-like quantum circuit for period finding.
    (Placeholder - a full implementation of modular exponentiation is complex and depends on the specific N and a)
    """
    if gcd(a, N) != 1:
        # Shor's algorithm requires 'a' to be coprime to 'N'
        print(f"Warning: a={a} and N={N} are not coprime. Shor's algorithm won't work correctly for factorization.")
        # For demonstration it proceeds but results will not be meaningful for factoring

    n_count = 2 * N.bit_length() # Number of counting qubits
    n_value = N.bit_length()     # Number of qubits for the value register

    qc = QuantumCircuit(n_count + n_value, n_count)

    # Initialize counting qubits to |+> state
    qc.h(range(n_count))

    # Initialize value register to |1> (first qubit of the value register)
    qc.x(n_count)

    # Placeholder for Controlled-U operations (modular exponentiation)
    # The actual implementation of C-U_a,N |y> = |(a*y) mod N> is complex and depends on the specific N and a. For this example, we apply some
    # controlled phase gates and CNOTs to induce superposition and entanglement, allowing the QFT to produce varied results for testing the adaptive shots logic.
    # This is NOT a correct modular exponentiation circuit for Shor's algorithm.
    for control_qubit_idx in range(n_count):
        # Apply a controlled-phase gate (CP)
        # Angle chosen here is arbitrary for generating some quantum dynamics.
        qc.cp(np.pi / (2**(n_count - 1 - control_qubit_idx)), control_qubit_idx, n_count + 0)
        # Add a CNOT to interact with another value qubit if available
        if n_value > 1 and n_count + 1 < qc.num_qubits:
            qc.cx(control_qubit_idx, n_count + 1)

    # Applys inverse Quantum Fourier Transform to the counting qubits
    qc.append(QFT(n_count, inverse=True), range(n_count))

    # Measures the counting qubits
    qc.measure(range(n_count), range(n_count))

    # Simulates the circuit
    simulator = Aer.get_backend('qasm_simulator')
    transpiled_qc = transpile(qc, simulator)
    job = simulator.run(transpiled_qc, shots=shots)
    result = job.result()
    counts = result.get_counts(qc)

    return counts

def analyze_period(counts, N):
    """
    Analyzes measurement counts to estimate the period r.
    This is a simplified placeholder for Shor's period finding, as the quantum circuit itself is a placeholder.
    It attempts to find a likely period based on the most frequent measurement outcome using a heuristic.
    """
    if not counts:
        return None

    # Get the most frequent outcome
    most_common_outcome = max(counts, key=counts.get)
    decimal_outcome = int(most_common_outcome, 2) # Converts binary string to integer

    # n_count was defined as 2 * N.bit_length()
    n_count = 2 * N.bit_length()

    try:
        # The phase is decimal_outcome / (2**n_count), looking for a fraction s/r that approximates this phase
        # Use fractions.Fraction to approximate phase and find the denominator
        # Limit denominator to N to avoid large periods from noisy data
        phase_approx = Fraction(decimal_outcome, 2**n_count).limit_denominator(N)
        r_candidate = phase_approx.denominator

        # In a real Shor's, it would validate if a^(r_candidate) % N == 1
        # For this placeholder, a non-trivial period is considered a 'found' period
        if r_candidate > 1 and r_candidate < N:
            return r_candidate
        else:
            return None
    except ZeroDivisionError:
        return None
    except Exception as e:
        print(f"Error in analyze_period: {e}")
        return None

def adaptive_shots_shor(N, a, max_shots=10000):
    """
    Optimized Shor's algorithm with adaptive number of shots.
    Stops early when confident in period estimation.
    """
    shots_per_batch = 100
    accumulated_counts = {}
    total_shots = 0

    while total_shots < max_shots:
        # Runs batch
        batch_counts = run_shor_circuit(N, a, shots_per_batch)

        # Accumulates results
        for outcome, count in batch_counts.items():
            accumulated_counts[outcome] = accumulated_counts.get(outcome, 0) + count

        total_shots += shots_per_batch

        # Analyzes current data
        period = analyze_period(accumulated_counts, N)

        if period:
            # Checks confidence level
            confidence = calculate_confidence(accumulated_counts, period)
            if confidence > 0.95:
                print(f"Stopped after {total_shots} shots with confidence {confidence:.2%}")
                return period

        # Updates shots per batch adaptively
        if total_shots < 1000:
            shots_per_batch = 200
        elif total_shots < 5000:
            shots_per_batch = 500
        else:
            shots_per_batch = 1000

    return analyze_period(accumulated_counts, N)

def calculate_confidence(counts, candidate_period):
    #Calculate confidence in candidate period based on measurement distribution
    total = sum(counts.values())
    expected_peaks = [i/total for i in range(0, total, candidate_period)]
    # Simplified confidence calculation
    return min(0.95, len(expected_peaks) / (2**3))  # Placeholder

#Executes function
adaptive_shots_shor(5, 3, max_shots=10000)

print("Program took %s seconds to run" % (time.monotonic() - start_time))

/tmp/ipykernel_12151/4279405166.py:59: DeprecationWarning: The class ``qiskit.circuit.library.basis_change.qft.QFT`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. ('Use qiskit.circuit.library.QFTGate or qiskit.synthesis.qft.synth_qft_full instead, for access to all previous arguments.',)
  qc.append(QFT(n_count, inverse=True), range(n_count))


Program took 1778106778.5818558 seconds to run
